In [4]:
import pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import numpy as np

from utils import Optimizer, enhance, get_importance_score, Pipeline, MeanEncoder, SelectColumns

In [5]:
data = pd.read_csv("./data/train.csv", index_col=0)

to_lag = ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]
to_drop = ["E7", "V10", "S3", "M1", "M13", "M14", "M6", "V9"]  #缺失值多，直接删除列

#处理bool列
for i in range(1, 10):
    data[f"D{i}"] = data[f"D{i}"] != 0
    data[f"D{i}"] = data[f"D{i}"].astype("category")

#处理收益率列
for i in to_lag:
    data[f"lag_{i}"] = data[i].shift(1)

data["target"] = (data["forward_returns"] - data["risk_free_rate"]) > 0  #使用指数收益率是否大于无风险利率作为预测值
data

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns,lag_forward_returns,lag_risk_free_rate,lag_market_forward_excess_returns,target
date_id,,,,,,,,,,,,,,,,,,,,,
0,False,False,False,True,True,False,False,False,True,NaN,...,NaN,NaN,NaN,-0.002421,0.000301,-0.003038,NaN,NaN,NaN,False
1,False,False,False,True,True,False,False,False,True,NaN,...,NaN,NaN,NaN,-0.008495,0.000303,-0.009114,-0.002421,0.000301,-0.003038,False
2,False,False,False,True,False,False,False,False,True,NaN,...,NaN,NaN,NaN,-0.009624,0.000301,-0.010243,-0.008495,0.000303,-0.009114,False
3,False,False,False,True,False,False,False,False,False,NaN,...,NaN,NaN,NaN,0.004662,0.000299,0.004046,-0.009624,0.000301,-0.010243,True
4,False,False,False,True,False,False,False,False,False,NaN,...,NaN,NaN,NaN,-0.011686,0.000299,-0.012301,0.004662,0.000299,0.004046,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9016,False,False,False,True,False,False,False,False,False,1.493117,...,-0.327455,0.083995,-0.380452,-0.000015,0.000151,-0.000477,0.010401,0.000152,0.009936,False
9017,False,False,False,True,False,False,False,False,False,1.490889,...,-0.372979,0.094246,-0.427355,-0.005199,0.000150,-0.005661,-0.000015,0.000151,-0.000477,False
9018,False,False,False,True,False,True,False,False,False,1.488667,...,-0.282024,0.090608,-0.381337,0.005930,0.000150,0.005467,-0.005199,0.000150,-0.005661,True


In [6]:
windows = [126]

opt = Optimizer(
    dataset=data,
    primitives=[
        enhance.RollingMin(windows),
        enhance.RollingMax(windows),
        enhance.RollingMean(windows),
        enhance.RollingStd(windows),
        enhance.RollingCountTrue(windows),
        enhance.RollingCountAboveMean(windows),
        enhance.RollingTrend(windows),
        enhance.RollingMaxConsecutiveTrue(windows),
        enhance.RollingMaxConsecutivePositives(windows),
        enhance.RollingNumSinceLastTrue(windows),
        enhance.Lag(),
        enhance.Diff(),
        enhance.PctChange(),
    ],
    drop_columns=to_drop + to_lag,
    cut_off=5400
)

RollingMin处理的列: ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M10', 'M11', 'M12', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'lag_forward_returns', 'lag_risk_free_rate', 'lag_market_forward_excess_returns']
RollingMax处理的列: ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M10', 'M11', 'M12', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S4', 'S5

In [7]:
#Optimizer初始化后可以得到构造完成的x与y
opt.x.shape, opt.y.shape

((5400, 925), (5400,))

In [8]:
"""
计算基准策略的score
hold: 全程持仓为1
prev: 前一天涨则当日持仓为1，否则为0
rand: 当日持仓为随机的0或1
"""
opt.baseline()

{'hold': np.float64(0.9202675293325906),
 'prev': np.float64(0.256236083563701),
 'rand': np.float64(0.3596991257746469)}

In [9]:
score_df, _, _ = get_importance_score(opt.x, opt.y)
score_df

计算Importance
计算Null Importance


100%|██████████| 80/80 [04:49<00:00,  3.62s/it]


,feature,split_score,gain_score,split_rank,gain_rank,rank
665,lag_risk_free_rate_MaxConsPos_R126,2.397895,4.885598,923.0,925.0,924.0
324,V7_Mean_R126,3.075775,3.729034,925.0,923.0,924.0
42,M17,2.535679,3.293769,924.0,920.0,922.0
600,E3_MaxConsPos_R126,2.197225,3.617070,921.0,922.0,921.5
244,V7_Max_R126,2.302585,3.168410,922.0,919.0,920.5
...,...,...,...,...,...,...
91,E11_Min_R126,-23.025851,-23.025851,70.5,70.5,70.5
6,D7,-23.025851,-23.025851,70.5,70.5,70.5
2,D3,-23.025851,-23.025851,70.5,70.5,70.5
1,D2,-23.025851,-23.025851,70.5,70.5,70.5


### 设置窗口列表

例如63表示使用最近63条数据训练模型，然后使用该模型在测试集中进行预测

若设置为0表示使用之前所有历史数据训练模型

In [10]:
opt.look_back_list = [63, 126, 252]

### 设置选取的模型列表

模型要满足

```py
class Model(Protocol):
    def fit(self, x_train, y_train): # x_train的length就是look_back的值
        ...

    def predict(self, x_test) -> pd.DataFrame:
        ...
```

predict预测结果可以是bool也可以是数值

In [11]:
opt.model_clz_list = [
    XGBClassifier,
    DecisionTreeClassifier,
    RandomForestClassifier,
    LGBMClassifier,
]

### 设置weight_func列表

weight_func将模型预测值series转化为权重series

例如对于分类模型，使用identical表示当预测值为True（即预测指数收益率大于无风险利率）时，持仓为1，否则持仓为0

In [12]:
def identical(y):
    return y


def double(y):
    return y * 2


opt.weight_func_list = [identical, double]

### 设置Pipeline

模型训练前对数据进行处理

```py

...

x_train, y_train = self.pipeline.fit_transform(x_train, y_train)
x_test, y_test = self.pipeline.transform(x_test, y_test)

model = model_clz()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

...

```

In [13]:
opt.pipeline = Pipeline([
    SelectColumns(score_df[:50]["feature"]),
    MeanEncoder()
])

In [14]:
opt.score()

100%|██████████| 24/24 [00:21<00:00,  1.10it/s]


,model,look_back,weight_func,score
0,XGBClassifier,63,identical,0.454963
1,XGBClassifier,63,double,0.562422
2,XGBClassifier,126,identical,0.332016
3,XGBClassifier,126,double,0.863965
4,XGBClassifier,252,identical,0.673530
5,XGBClassifier,252,double,1.053298
6,DecisionTreeClassifier,63,identical,0.804287
7,DecisionTreeClassifier,63,double,0.672163
8,DecisionTreeClassifier,126,identical,0.410085
9,DecisionTreeClassifier,126,double,0.257688


### 使用回归模型

In [17]:
data = pd.read_csv("./data/train.csv", index_col=0)

to_lag = ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]
to_drop = ["E7", "V10", "S3", "M1", "M13", "M14", "M6", "V9"]  #缺失值多，直接删除列

#处理bool列
for i in range(1, 10):
    data[f"D{i}"] = data[f"D{i}"] != 0
    data[f"D{i}"] = data[f"D{i}"].astype("category")

#处理收益率列
for i in to_lag:
    data[f"lag_{i}"] = data[i].shift(1)

data["target"] = data["forward_returns"] - data["risk_free_rate"] #使用指数收益率减无风险利率的值作为预测值

#修改weight_func实现
def to_weight(arr):
    """
    arr 模型预测的指数收益率与无风险利率之差
    根据阈值对数组值进行分类：
    - 大于90%分位数 -> 2
    - 大于平均值且小于等于90%分位数 -> 1
    - 其他 -> 0
    """
    arr = arr.copy()  # 避免修改原始数组
    mean_val = np.mean(arr)
    p90 = np.percentile(arr, 90)

    # 创建条件掩码
    mask_gt_p90 = arr > p90
    mask_gt_mean_le_p90 = (arr > mean_val) & (arr <= p90)

    # 应用转换
    arr[mask_gt_p90] = 2
    arr[mask_gt_mean_le_p90] = 1
    arr[~(mask_gt_p90 | mask_gt_mean_le_p90)] = 0
    return arr


opt = Optimizer(
    dataset=data,
    primitives=[],
    drop_columns=to_drop + to_lag,
    cut_off=5400
)

opt.report_one(
    model_clz=XGBRegressor,
    look_back=252,
    weight_func=to_weight,
    pipeline=Pipeline([MeanEncoder()])
)

,sharpe,hold_sharpe,prev_sharpe,rand_sharpe
5274:5400,0.285671,3.403157,0.342322,0.835834
5148:5274,0.712459,-0.903425,-1.389761,-0.461109
5022:5148,0.036209,1.783775,0.378242,0.785187
4896:5022,3.858988,2.858746,1.560802,1.021711
4770:4896,0.998684,-0.000431,1.121110,0.190301
4644:4770,0.095500,1.030584,-0.088293,0.702825
4518:4644,-1.148462,-1.356372,-1.224475,-0.843427
4392:4518,2.070416,-0.292491,-0.063249,0.154624
4266:4392,0.673917,1.758864,1.669427,0.865736
avg,0.842598,0.920268,0.256236,0.361298
